# Заняття 03 — SQL для Data Engineering

## Основні цілі заняття:
* Освіжити **window functions** і **CTE**.
* Показати DDL у DuckDB: типи, `CHECK`, `NOT NULL` — примусово задати схему.
* Побачити виконання запиту через **EXPLAIN ANALYZE**.
* Перекинути міст між DuckDB і Polars через Apache Arrow.
* Познайомитися з **dbt** як парадигмою: SQL як код, що тестується й документується.

* **Датасет:** NYC TLC Yellow Taxi Trip Records, January 2024 — **той самий файл, що в занятті 02**
* **Формат:** Parquet (~100 MB, ~3 млн рядків)
* **База даних:** `taxi_dwh.duckdb` — постійна (не in-memory)

Структура ноутбука:
1. Читання Parquet напряму (schema-on-read) + SUMMARIZE
2. Читання remote Parquet через httpfs
3. DDL: типізована Bronze-таблиця + CHECK-обмеження
4. Window functions + CTE: відповідаємо на бізнес-питання
5. DuckDB → Polars: zero-copy через Apache Arrow
6. EXPLAIN ANALYZE: як DuckDB виконує запит
7. COPY: записуємо Silver у Parquet
8. DuckDB-специфіка: views, nested types, prepared statements

## 1. Читання Parquet напряму (schema-on-read)

DuckDB читає Parquet **без DDL** — типи виводяться з footer файлу.
Цей підхід зручний для швидкого EDA, але не гарантує якість даних.

In [10]:
from pathlib import Path
import duckdb
from icecream import ic

REMOTE = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
ZONE_PARQUET = Path("../../../data/lesson-02/reference/taxi_zone_lookup.parquet")

con = duckdb.connect("taxi_dwh.duckdb")

### DESCRIBE — схема без завантаження даних

In [11]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{REMOTE}')")

┌───────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│      column_name      │ column_type │  null   │   key   │ default │  extra  │
│        varchar        │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ VendorID              │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ tpep_pickup_datetime  │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ tpep_dropoff_datetime │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ passenger_count       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ trip_distance         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ RatecodeID            │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ store_and_fwd_flag    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ PULocationID          │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ DOLocationID          │ INTEGER     │ 

### SUMMARIZE — статистика по кожній колонці

Зверніть увагу на `min` у колонці `fare_amount` — він **від'ємний**.
Це сигнал про DQ-проблему, з якою ми розберемося в секції 3.

In [12]:
con.sql(f"SUMMARIZE SELECT * FROM read_parquet('{REMOTE}')")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────────────┬─────────────┬─────────────────────┬─────────────────────┬───────────────┬────────────────────────────┬─────────────────────┬────────────────────────────┬────────────────────────────┬────────────────────────────┬─────────┬─────────────────┐
│      column_name      │ column_type │         min         │         max         │ approx_unique │            avg             │         std         │            q25             │            q50             │            q75             │  count  │ null_percentage │
│        varchar        │   varchar   │       varchar       │       varchar       │     int64     │          varchar           │       varchar       │          varchar           │          varchar           │          varchar           │  int64  │  decimal(9,2)   │
├───────────────────────┼─────────────┼─────────────────────┼─────────────────────┼───────────────┼────────────────────────────┼─────────────────────┼────────────────────────────┼───────────────────────

## 2. Читання remote Parquet через httpfs

DuckDB вміє читати Parquet **напряму з HTTP/S3** — жодного попереднього
завантаження. `httpfs` робить range-запити: завантажується тільки footer
(схема) + потрібні row groups, а не весь файл.

In [13]:
con.sql("INSTALL httpfs; LOAD httpfs;")

In [14]:
con.sql(f"SELECT count(*) AS total_rows FROM read_parquet('{REMOTE}')")

┌────────────┐
│ total_rows │
│   int64    │
├────────────┤
│    2964624 │
└────────────┘

Локальний файл — офлайн-fallback для тих, хто без інтернету:
```python
con.sql(f"SELECT count(*) FROM read_parquet('{REMOTE}')")
```

## 3. DDL: типізована Bronze-таблиця + CHECK

`schema-on-read` не перевіряє значення. DDL дозволяє:
- задати **точні типи** (DECIMAL замість DOUBLE для грошей)
- поставити **NOT NULL** там, де відсутність значення — помилка
- задати **CHECK**-обмеження, які DuckDB перевіряє при INSERT

Колонки з відомими null-ами в 2024-даних (`passenger_count`, `RatecodeID`,
`Airport_fee`, `congestion_surcharge`) — без NOT NULL.

In [15]:
con.sql("""
CREATE OR REPLACE TABLE bronze_yellow_trips (
    vendor_id             INTEGER,
    tpep_pickup_datetime  TIMESTAMP    NOT NULL,
    tpep_dropoff_datetime TIMESTAMP    NOT NULL,
    passenger_count       SMALLINT,
    trip_distance         DOUBLE,
    ratecode_id           SMALLINT,
    store_and_fwd_flag    VARCHAR,
    pu_location_id        USMALLINT    NOT NULL,
    do_location_id        USMALLINT    NOT NULL,
    payment_type          SMALLINT,
    fare_amount           DECIMAL(10,2) NOT NULL,
    extra                 DECIMAL(10,2),
    mta_tax               DECIMAL(10,2),
    tip_amount            DECIMAL(10,2),
    tolls_amount          DECIMAL(10,2),
    improvement_surcharge DECIMAL(10,2),
    total_amount          DECIMAL(10,2) NOT NULL,
    congestion_surcharge  DECIMAL(10,2),
    airport_fee           DECIMAL(10,2),
    CHECK (fare_amount >= 0),
    CHECK (trip_distance >= 0),
    CHECK (passenger_count BETWEEN 0 AND 9 OR passenger_count IS NULL)
)
""")

### INSERT без фільтра → CHECK-помилка

Вставляємо всі рядки, включно з `fare_amount < 0`.
DuckDB відкатить весь INSERT — транзакційна семантика.

In [16]:
try:
    con.sql(f"""
    INSERT INTO bronze_yellow_trips
    SELECT
        VendorID, tpep_pickup_datetime, tpep_dropoff_datetime,
        passenger_count, trip_distance, RatecodeID, store_and_fwd_flag,
        PULocationID, DOLocationID, payment_type,
        fare_amount, extra, mta_tax, tip_amount, tolls_amount,
        improvement_surcharge, total_amount, congestion_surcharge, Airport_fee
    FROM read_parquet('{REMOTE}')
    """)
except Exception as e:
    print(f"CHECK порушено — INSERT відкатано:\n{e}")

CHECK порушено — INSERT відкатано:
Constraint Error: CHECK constraint failed on table bronze_yellow_trips with expression CHECK((fare_amount >= 0))


### INSERT із фільтром → Bronze матеріалізована

In [17]:
con.sql(f"""
INSERT INTO bronze_yellow_trips
SELECT
    VendorID, tpep_pickup_datetime, tpep_dropoff_datetime,
    passenger_count, trip_distance, RatecodeID, store_and_fwd_flag,
    PULocationID, DOLocationID, payment_type,
    fare_amount, extra, mta_tax, tip_amount, tolls_amount,
    improvement_surcharge, total_amount, congestion_surcharge, Airport_fee
FROM read_parquet('{REMOTE}')
WHERE fare_amount >= 0
  AND trip_distance >= 0
""")

ic(con.sql("SELECT count(*) FROM bronze_yellow_trips").fetchone()[0])

ic| con.sql("SELECT count(*) FROM bronze_yellow_trips").fetchone()[0]: 2927176


2927176

Завантажимо також довідник зон — він знадобиться в наступних секціях.

In [18]:
con.sql(f"""
CREATE OR REPLACE TABLE taxi_zones AS
SELECT * FROM read_parquet('{ZONE_PARQUET}')
""")
ic(con.sql("SELECT count(*) FROM taxi_zones").fetchone()[0])

IOException: IO Error: No files found that match the pattern "../../../data/lesson-02/reference/taxi_zone_lookup.parquet"

LINE 3: SELECT * FROM read_parquet('../../../data/lesson-02/reference/taxi_zone_l...
                      ^

In [ ]:
# Check the correct path to the taxi zone lookup file
import os
from pathlib import Path

# List available paths to find the correct location
print("Current working directory:", os.getcwd())
print("\nSearching for taxi_zone_lookup.parquet...")

# Try to find the file
for root, dirs, files in os.walk(".."):
    for file in files:
        if "taxi_zone" in file.lower():
            full_path = os.path.join(root, file)
            print(f"Found: {full_path}")

# Once you find the correct path, update ZONE_PARQUET variable
# For example, if the file is in a different location:
# ZONE_PARQUET = Path("path/to/taxi_zone_lookup.parquet")

## 4. Window functions + CTE: відповідаємо на бізнес-питання

**Window function** vs **GROUP BY**: GROUP BY колапсує рядки до груп;
window function зберігає всі рядки й додає агреговане значення поряд.

### 4.1 TOP-5 найдорожчих поїздок по зоні відправлення (ROW_NUMBER + QUALIFY)

`QUALIFY` — DuckDB-спосіб фільтрувати за результатом window function
прямо у SELECT, без вкладеного запиту.

In [ ]:
con.sql("""
SELECT
    pu_location_id,
    tpep_pickup_datetime,
    fare_amount,
    total_amount,
    ROW_NUMBER() OVER (
        PARTITION BY pu_location_id
        ORDER BY total_amount DESC
    ) AS rank_in_zone
FROM bronze_yellow_trips
QUALIFY rank_in_zone <= 5
ORDER BY pu_location_id, rank_in_zone
LIMIT 20
""")

┌────────────────┬──────────────────────┬───────────────┬───────────────┬──────────────┐
│ pu_location_id │ tpep_pickup_datetime │  fare_amount  │ total_amount  │ rank_in_zone │
│     uint16     │      timestamp       │ decimal(10,2) │ decimal(10,2) │    int64     │
├────────────────┼──────────────────────┼───────────────┼───────────────┼──────────────┤
│              1 │ 2024-01-05 23:48:41  │        267.94 │        323.33 │            1 │
│              1 │ 2024-01-18 02:40:41  │        250.00 │        251.00 │            2 │
│              1 │ 2024-01-16 14:35:08  │        193.00 │        232.80 │            3 │
│              1 │ 2024-01-30 13:51:05  │        164.00 │        215.15 │            4 │
│              1 │ 2024-01-17 07:17:55  │        208.00 │        209.00 │            5 │
│              2 │ 2024-01-08 22:30:16  │         70.00 │         90.94 │            1 │
│              2 │ 2024-01-04 05:29:47  │         36.60 │         46.90 │            2 │
│              2 │ 20

### 4.2 Кумулятивна виручка по годині доби (SUM OVER)

In [ ]:
con.sql("""
WITH hourly AS (
    SELECT
        datepart('hour', tpep_pickup_datetime) AS hour_of_day,
        sum(fare_amount)                        AS hourly_revenue
    FROM bronze_yellow_trips
    GROUP BY 1
)
SELECT
    hour_of_day,
    hourly_revenue,
    SUM(hourly_revenue) OVER (ORDER BY hour_of_day) AS cumulative_revenue
FROM hourly
ORDER BY hour_of_day
""")

┌─────────────┬────────────────┬────────────────────┐
│ hour_of_day │ hourly_revenue │ cumulative_revenue │
│    int64    │ decimal(38,2)  │   decimal(38,2)    │
├─────────────┼────────────────┼────────────────────┤
│           0 │     1547287.74 │         1547287.74 │
│           1 │      956726.12 │         2504013.86 │
│           2 │      629928.93 │         3133942.79 │
│           3 │      461087.40 │         3595030.19 │
│           4 │      386031.94 │         3981062.13 │
│           5 │      508500.65 │         4489562.78 │
│           6 │      907221.47 │         5396784.25 │
│           7 │     1567011.53 │         6963795.78 │
│           8 │     2087787.31 │         9051583.09 │
│           9 │     2306122.80 │        11357705.89 │
│           · │          ·     │             ·      │
│           · │          ·     │             ·      │
│           · │          ·     │             ·      │
│          14 │     3505122.81 │        26033134.40 │
│          15 │     3601172.

### 4.3 Зміна кількості поїздок годину-до-години (LAG)

In [ ]:
con.sql("""
WITH hourly_trips AS (
    SELECT
        datepart('hour', tpep_pickup_datetime) AS hour_of_day,
        count(*)                               AS trip_count
    FROM bronze_yellow_trips
    GROUP BY 1
)
SELECT
    hour_of_day,
    trip_count,
    LAG(trip_count) OVER (ORDER BY hour_of_day) AS prev_hour_trips,
    trip_count - LAG(trip_count) OVER (ORDER BY hour_of_day) AS delta
FROM hourly_trips
ORDER BY hour_of_day
""")

┌─────────────┬────────────┬─────────────────┬────────┐
│ hour_of_day │ trip_count │ prev_hour_trips │ delta  │
│    int64    │   int64    │      int64      │ int64  │
├─────────────┼────────────┼─────────────────┼────────┤
│           0 │      77694 │            NULL │   NULL │
│           1 │      52684 │           77694 │ -25010 │
│           2 │      36783 │           52684 │ -15901 │
│           3 │      24252 │           36783 │ -12531 │
│           4 │      16304 │           24252 │  -7948 │
│           5 │      18378 │           16304 │   2074 │
│           6 │      40873 │           18378 │  22495 │
│           7 │      82936 │           40873 │  42063 │
│           8 │     116128 │           82936 │  33192 │
│           9 │     127734 │          116128 │  11606 │
│           · │        ·   │             ·   │    ·   │
│           · │        ·   │             ·   │    ·   │
│           · │        ·   │             ·   │    ·   │
│          14 │     180815 │          167880 │  

### 4.4 CTE: зони з посадками, але без висадок (DQ-паттерн)

Anti-join через CTE — класичний прийом для виявлення «осиротілих» значень.
Якщо зона є у `PULocationID`, але ніколи не зустрічається у `DOLocationID` —
це може бути артефакт даних або реальне обмеження (аеропорт-виїзд тільки).

In [ ]:
con.sql("""
WITH pu_zones AS (
    SELECT DISTINCT pu_location_id AS location_id FROM bronze_yellow_trips
),
do_zones AS (
    SELECT DISTINCT do_location_id AS location_id FROM bronze_yellow_trips
)
SELECT p.location_id
FROM pu_zones p
LEFT JOIN do_zones d ON p.location_id = d.location_id
WHERE d.location_id IS NULL
ORDER BY 1
""")

┌─────────────┐
│ location_id │
│   uint16    │
├─────────────┤
│         199 │
└─────────────┘

## 5. DuckDB → Polars: zero-copy через Apache Arrow

DuckDB **не має stored procedures** за дизайном: процедурна логіка виноситься
назовні. Python — природний вибір. Перехід відбувається через Apache Arrow —
**zero-copy**: дані не копіюються у пам'яті, лише передається вказівник.

In [ ]:
# Eager bridge: DuckDB → Polars DataFrame через .pl()
q = """
SELECT pu_location_id, fare_amount, tip_amount, total_amount
FROM bronze_yellow_trips
WHERE fare_amount > 20
"""
df_duck_db = con.sql(q)


In [ ]:
df_duck_db.shape


(744560, 4)

In [ ]:
df_pl = df_duck_db.pl()

In [ ]:
df_pl.shape

(744560, 4)

## 6. EXPLAIN ANALYZE: як DuckDB виконує запит

`EXPLAIN ANALYZE` запускає запит і повертає фактичний план із статистикою
виконання. Повний текст плану — у другому стовпці першого рядка результату.

### E1. Projection + sargable filter → predicate pushdown

DuckDB пушить фільтр і проєкцію безпосередньо у Parquet-сканування.
Шукаємо у плані: `Rows Scanned` та які колонки читаються.

In [ ]:
print(con.sql("""
EXPLAIN ANALYZE
SELECT pu_location_id, fare_amount, tpep_pickup_datetime
FROM bronze_yellow_trips
WHERE tpep_pickup_datetime >= '2024-01-15'
  AND tpep_pickup_datetime <  '2024-01-16'
""").fetchone()[1])

┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││    Query Profiling Information    ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
 EXPLAIN ANALYZE SELECT pu_location_id, fare_amount, tpep_pickup_datetime FROM bronze_yellow_trips WHERE tpep_pickup_datetime >= '2024-01-15'   AND tpep_pickup_datetime <  '2024-01-16' 
┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││         HTTPFS HTTP Stats         ││
││                                   ││
││            in: 0 bytes            ││
││            out: 0 bytes           ││
││              #HEAD: 0             ││
││              #GET: 0              ││
││              #PUT: 0              ││
││              #POST: 0             ││
││             #DELETE: 0            ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
┌────────────────────────────────────────────────┐
│┌─────────────────────────────────────────

### E2. Partition pruning

Імітуємо партиоцінування за директоріями

In [ ]:
con.sql("""
COPY (
    SELECT pu_location_id, fare_amount, tpep_pickup_datetime,
           CAST(tpep_pickup_datetime AS DATE) AS pickup_date
    FROM bronze_yellow_trips
    WHERE tpep_pickup_datetime >= '2024-01-01'
      AND tpep_pickup_datetime <  '2024-02-01'
) TO '../../data/lesson-03/silver/partitioned_by_date' (FORMAT PARQUET, PARTITION_BY (pickup_date), OVERWRITE_OR_IGNORE TRUE)
""")

Фільтр не по-партиції

In [ ]:
print(con.sql("""
EXPLAIN ANALYZE
SELECT count(*) FROM read_parquet('../../data/lesson-03/silver/partitioned_by_date/**/*.parquet', hive_partitioning = true)
WHERE fare_amount > 100
""").fetchone()[1])

┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││    Query Profiling Information    ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
 EXPLAIN ANALYZE SELECT count(*) FROM read_parquet('../../data/lesson-03/silver/partitioned_by_date/**/*.parquet', hive_partitioning = true) WHERE fare_amount > 100 
┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││         HTTPFS HTTP Stats         ││
││                                   ││
││            in: 0 bytes            ││
││            out: 0 bytes           ││
││              #HEAD: 0             ││
││              #GET: 0              ││
││              #PUT: 0              ││
││              #POST: 0             ││
││             #DELETE: 0            ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
┌────────────────────────────────────────────────┐
│┌──────────────────────────────────────────────┐│
││          

Фільтр по-партиції

In [ ]:
print(con.sql("""
EXPLAIN ANALYZE
SELECT count(*) FROM read_parquet('../../data/lesson-03/silver/partitioned_by_date/**/*.parquet', hive_partitioning = true)
WHERE pickup_date = '2024-01-15'
""").fetchone()[1])

┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││    Query Profiling Information    ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
 EXPLAIN ANALYZE SELECT count(*) FROM read_parquet('../../data/lesson-03/silver/partitioned_by_date/**/*.parquet', hive_partitioning = true) WHERE pickup_date = '2024-01-15' 
┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││         HTTPFS HTTP Stats         ││
││                                   ││
││            in: 0 bytes            ││
││            out: 0 bytes           ││
││              #HEAD: 0             ││
││              #GET: 0              ││
││              #PUT: 0              ││
││              #POST: 0             ││
││             #DELETE: 0            ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
┌────────────────────────────────────────────────┐
│┌──────────────────────────────────────────────┐│
││ 

In [ ]:
print(con.sql("""
EXPLAIN ANALYZE
SELECT count(*) FROM read_parquet('../../data/lesson-03/silver/partitioned_by_date/**/*.parquet', hive_partitioning = true)
WHERE pickup_date >= '2024-01-10' AND pickup_date < '2024-01-16'
""").fetchone()[1])

┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││    Query Profiling Information    ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
 EXPLAIN ANALYZE SELECT count(*) FROM read_parquet('../../data/lesson-03/silver/partitioned_by_date/**/*.parquet', hive_partitioning = true) WHERE pickup_date >= '2024-01-10' AND pickup_date < '2024-01-16' 
┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││         HTTPFS HTTP Stats         ││
││                                   ││
││            in: 0 bytes            ││
││            out: 0 bytes           ││
││              #HEAD: 0             ││
││              #GET: 0              ││
││              #PUT: 0              ││
││              #POST: 0             ││
││             #DELETE: 0            ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
┌────────────────────────────────────────────────┐
│┌────────────────────

### E3. Hash join — яка таблиця є build side?

DuckDB будує hash-таблицю з **меншого** датасету.
`taxi_zones` (~260 рядків) — build side; `bronze_yellow_trips` (~3 млн) — probe side.
Це точний аналог **broadcast join** у Spark (Lesson 15).

In [ ]:
print(con.sql("""
EXPLAIN ANALYZE
SELECT z.Borough, count(*) AS trips, avg(t.fare_amount) AS avg_fare
FROM bronze_yellow_trips t
JOIN taxi_zones z ON t.pu_location_id = z.LocationID
GROUP BY z.Borough
""").fetchone()[1])

┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││    Query Profiling Information    ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
 EXPLAIN ANALYZE SELECT z.Borough, count(*) AS trips, avg(t.fare_amount) AS avg_fare FROM bronze_yellow_trips t JOIN taxi_zones z ON t.pu_location_id = z.LocationID GROUP BY z.Borough 
┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││         HTTPFS HTTP Stats         ││
││                                   ││
││            in: 0 bytes            ││
││            out: 0 bytes           ││
││              #HEAD: 0             ││
││              #GET: 0              ││
││              #PUT: 0              ││
││              #POST: 0             ││
││             #DELETE: 0            ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
┌────────────────────────────────────────────────┐
│┌──────────────────────────────────────────

### E3.5. Aggregate before join — допомагає чи оптимізатор вже знає?

CTE агрегує trips до ~260 рядків ДО join, щоб зменшити probe-сторону.
Порівняємо плани: якщо вони різні — ручний рефакторинг виграв;
якщо ідентичні — оптимізатор вже все зробив.

In [ ]:
print(con.sql("""
EXPLAIN ANALYZE
WITH per_zone AS (
    SELECT pu_location_id,
           count(*)         AS trips,
           sum(fare_amount) AS sum_fare      -- carry the components, not the avg
    FROM bronze_yellow_trips
    GROUP BY pu_location_id
)
SELECT z.Borough,
       sum(p.trips)                    AS total_trips,
       sum(p.sum_fare) / sum(p.trips)  AS borough_avg_fare   -- recompute the weighted avg
FROM per_zone p
JOIN taxi_zones z ON p.pu_location_id = z.LocationID
GROUP BY z.Borough
""").fetchone()[1])

┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││    Query Profiling Information    ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
 EXPLAIN ANALYZE WITH per_zone AS (     SELECT pu_location_id,            count(*)         AS trips,            sum(fare_amount) AS sum_fare      -- carry the components, not the avg     FROM bronze_yellow_trips     GROUP BY pu_location_id ) SELECT z.Borough,        sum(p.trips)                    AS total_trips,        sum(p.sum_fare) / sum(p.trips)  AS borough_avg_fare   -- recompute the weighted avg FROM per_zone p JOIN taxi_zones z ON p.pu_location_id = z.LocationID GROUP BY z.Borough 
┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││         HTTPFS HTTP Stats         ││
││                                   ││
││            in: 0 bytes            ││
││            out: 0 bytes           ││
││              #HEAD: 0             ││
││              #GET: 0  

## 7. COPY: записуємо Silver у Parquet

In [ ]:
Path("data/silver").mkdir(parents=True, exist_ok=True)

### Flat Silver — один Parquet-файл

In [ ]:
con.sql("""
COPY (
    SELECT
        vendor_id, tpep_pickup_datetime, tpep_dropoff_datetime,
        pu_location_id, do_location_id, payment_type,
        fare_amount, tip_amount, total_amount
    FROM bronze_yellow_trips
) TO '../../data/lesson-03/silver/yellow_trips.parquet' (FORMAT PARQUET)
""")
ic(Path("../../data/lesson-03/silver/yellow_trips.parquet").stat().st_size / 1e6)

ic| Path("../../data/lesson-03/silver/yellow_trips.parquet").stat().st_size / 1e6: 51.176881


51.176881

### Partitioned Silver — Hive layout по borough

`PARTITION_BY` створює ієрархію папок `borough=Manhattan/`, `borough=Queens/` тощо.
Downstream-запити можуть пушити фільтр на рівень partition — без читання зайвих файлів.

In [ ]:
con.sql("""
COPY (
    SELECT
        t.vendor_id, t.tpep_pickup_datetime, t.tpep_dropoff_datetime,
        t.pu_location_id, t.do_location_id, t.payment_type,
        t.fare_amount, t.tip_amount, t.total_amount,
        z.Borough AS borough
    FROM bronze_yellow_trips t
    JOIN taxi_zones z ON t.pu_location_id = z.LocationID
) TO '../../data/lesson-03/silver/partitioned' (FORMAT PARQUET, PARTITION_BY (borough), OVERWRITE_OR_IGNORE TRUE)
""")
ic(sorted(p.name for p in Path("../../data/lesson-03/silver/partitioned").iterdir()))

ic| sorted(p.name for p in Path("../../data/lesson-03/silver/partitioned").iterdir()): ['borough=Bronx',
                                                                                        'borough=Brooklyn',
                                                                                        'borough=EWR',
                                                                                        'borough=Manhattan',
                                                                                        'borough=Queens',
                                                                                        'borough=Staten%20Island',
                                                                                        'borough=Unknown',
                                                                                        'borough=__HIVE_DEFAULT_PARTITION__']


['borough=Bronx',
 'borough=Brooklyn',
 'borough=EWR',
 'borough=Manhattan',
 'borough=Queens',
 'borough=Staten%20Island',
 'borough=Unknown',
 'borough=__HIVE_DEFAULT_PARTITION__']

## 8. DuckDB-специфіка

### CREATE VIEW vs CREATE TABLE

`VIEW` — збережений SQL-запит, виконується щоразу при зверненні.
`TABLE` — матеріалізовані дані на диску.
Ця різниця відображається в dbt як `materialized: view` vs `materialized: table` —
той самий вибір, але в керованому pipeline.

In [ ]:
con.sql("""
CREATE OR REPLACE VIEW v_long_trips AS
SELECT pu_location_id, do_location_id, trip_distance, fare_amount
FROM bronze_yellow_trips
WHERE trip_distance > 20
""")
ic(con.sql("SELECT count(*) FROM v_long_trips").fetchone()[0])

ic| con.sql("SELECT count(*) FROM v_long_trips").fetchone()[0]: 28960


28960

In [ ]:
# View — це просто збережений текст запиту
con.sql("SELECT sql FROM duckdb_views() WHERE view_name = 'v_long_trips'")

┌────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                                                                        sql                                                                         │
│                                                                      varchar                                                                       │
├────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ CREATE VIEW v_long_trips AS SELECT pu_location_id, do_location_id, trip_distance, fare_amount FROM bronze_yellow_trips WHERE (trip_distance > 20); │
└────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┘

### Nested types — one-liner

DuckDB підтримує `STRUCT`, `LIST`, `MAP` нативно.
Корисно для зберігання JSON-подібних структур без втрати типів.

In [ ]:
con.sql("""
SELECT
    {'vendor': vendor_id, 'pickup': pu_location_id} AS trip_struct,
    [fare_amount, tip_amount, total_amount]          AS amount_list
FROM bronze_yellow_trips
LIMIT 3
""")

┌──────────────────────────────────────────┬──────────────────────┐
│               trip_struct                │     amount_list      │
│ struct(vendor integer, pickup usmallint) │   decimal(10,2)[]    │
├──────────────────────────────────────────┼──────────────────────┤
│ {'vendor': 2, 'pickup': 186}             │ [17.70, 0.00, 22.70] │
│ {'vendor': 1, 'pickup': 140}             │ [10.00, 3.75, 18.75] │
│ {'vendor': 1, 'pickup': 236}             │ [23.30, 3.00, 31.30] │
└──────────────────────────────────────────┴──────────────────────┘

### Prepared statements — bind, don't f-string

Параметр `$1` (або `?`) — це **bind parameter**: значення передається окремо
від SQL-тексту. DuckDB перевіряє тип параметра і не допускає SQL injection.

Правило: **ніколи не підставляйте значення через f-string у SQL-запити**.
Завжди використовуйте bind-параметри.

In [ ]:
result = con.execute(
    "SELECT count(*) FROM bronze_yellow_trips WHERE pu_location_id = $1",
    [161]
)
ic(result.fetchone()[0])

ic| result.fetchone()[0]: 141749


141749

In [ ]:
# Порівняйте: правильно vs небезпечно
location_id = 161
# Правильно — bind parameter
safe_result = con.execute(
    "SELECT avg(fare_amount) FROM bronze_yellow_trips WHERE pu_location_id = $1",
    [location_id]
)
# Небезпечно — SQL injection можливий, якщо location_id від користувача
unsafe = con.sql(f"SELECT avg(fare_amount) FROM bronze_yellow_trips WHERE pu_location_id = {location_id}")

ic(safe_result.fetchone()[0])

ic| safe_result.fetchone()[0]: 15.581609182428094


15.581609182428094

### CLI та DBeaver — markdown showcase

Перед тим як відкрити DBeaver або CLI, **закриваємо з'єднання**:
DuckDB — **single-writer**: тільки одне з'єднання може писати одночасно.

In [ ]:
#con.close()

**DuckDB CLI** — запустити з терміналу:
```bash
duckdb taxi_dwh.duckdb
```
Команди у CLI: `.tables`, `.schema bronze_yellow_trips`, `.exit`

**DBeaver** підключається через **JDBC-драйвер DuckDB**:
- Driver: `DuckDB` (встановити через DBeaver Driver Manager)
- Connection URL: `jdbc:duckdb:/абсолютний/шлях/до/taxi_dwh.duckdb`
- Або відкрити у read-only mode: `jdbc:duckdb:/path/taxi_dwh.duckdb?access_mode=read_only`

У read-only mode можна мати кілька одночасних reader-з'єднань поверх writer.

## А далі?

Ми написали десятки SQL-запитів, побудували Bronze і Silver — вручну.
А тепер уявіть, що цей pipeline треба запускати щодня,
в певному порядку, з тестами на якість і з документацією для команди.

Як би ви це організували?